In [1]:
import pandas as pd
import numpy as np
import warnings

In [2]:
def fill_nearest_within_range(df, group_cols, target_col, window=10):
    df = df.copy()
    
    for group_values, group_df in df.groupby(group_cols):
        valid_idx = group_df[group_df[target_col].notna()].index.to_list()
        all_idx = group_df.index.to_list()

        index_pos_map = {idx: pos for pos, idx in enumerate(all_idx)}

        for idx in all_idx:
            if pd.isna(df.at[idx, target_col]):
                pos = index_pos_map[idx]
                start_pos = max(pos - window, 0)
                end_pos = min(pos + window, len(all_idx) - 1)
                candidates = [i for i in valid_idx if start_pos <= index_pos_map[i] <= end_pos]
                if not candidates:
                    continue

                candidate_dist = [(abs(index_pos_map[c] - pos), c) for c in candidates]
                min_dist, closest_idx = min(candidate_dist, key=lambda x: x[0])
                df.at[idx, target_col] = df.at[closest_idx, target_col]

                
    return df


In [ ]:
warnings.filterwarnings("ignore")

# --------------------
# 데이터 로드 및 전처리
# --------------------
data = pd.read_csv('test.csv', dtype={"evse_name": str, "station_location": str})

# UNIX timestamp → datetime (초 단위)
data["last_charge_end_time_ts"] = pd.to_datetime(data["last_charge_end_time_ts"], unit='s', errors='coerce').sort_values()

data = data.dropna(subset=["last_charge_end_time_ts", "evse_name", "station_location"])
data = data[data["last_charge_end_time_ts"] >= pd.Timestamp("2000-01-01")]




# 조합별 모든 time_idx 채우기 (Missing timestep 처리)
group_cols = ["station_location", "evse_name"]
unique_groups = data[group_cols].drop_duplicates()
df_store = pd.DataFrame()
# 그룹의 시간 범위 (min, max)
start_time = data["last_charge_end_time_ts"].min().floor('30T')
end_time = data["last_charge_end_time_ts"].max().ceil('30T')
end_time = end_time + pd.DateOffset(weeks=1)
#첫날 부터 30분씩 타임 슬롯 만들기
for idx, row in unique_groups.iterrows():
    loc = row["station_location"]
    evse = row["evse_name"]
    # 그룹별 유효 시간만 포함하여 DataFrame 생성

    # 해당 그룹의 데이터 필터링
    group_data = data[(data["station_location"] == loc) & (data["evse_name"] == evse)]  
    time_index = pd.DataFrame()
    # 30분 간격 시간 생성
    time_index['store_timestamp'] = pd.date_range(start=start_time, end=end_time, freq='30T')
    time_index['evse_type'] = group_data['evse_type'].iloc[0]
    time_index['station_location'] = group_data['station_location'].iloc[0]
    time_index['supports_discharge'] = group_data['supports_discharge'].iloc[0]
    time_index['weekday'] = time_index['store_timestamp'].dt.weekday
    time_index['month'] = time_index['store_timestamp'].dt.month
    time_index['evse_name'] = group_data['evse_name'].iloc[0]
     # 기존 df_store에 time_index를 이어 붙임
    df_store = pd.concat([df_store, time_index], ignore_index=True)

    print(time_index)


          store_timestamp evse_type station_location supports_discharge  \
0     2024-07-28 11:30:00        FC         CSCS2015                  n   
1     2024-07-28 12:00:00        FC         CSCS2015                  n   
2     2024-07-28 12:30:00        FC         CSCS2015                  n   
3     2024-07-28 13:00:00        FC         CSCS2015                  n   
4     2024-07-28 13:30:00        FC         CSCS2015                  n   
...                   ...       ...              ...                ...   
18472 2025-08-17 07:30:00        FC         CSCS2015                  n   
18473 2025-08-17 08:00:00        FC         CSCS2015                  n   
18474 2025-08-17 08:30:00        FC         CSCS2015                  n   
18475 2025-08-17 09:00:00        FC         CSCS2015                  n   
18476 2025-08-17 09:30:00        FC         CSCS2015                  n   

       weekday  month evse_name  
0            6      7         0  
1            6      7         0

In [4]:
df_store = df_store.sort_values('store_timestamp')
df_charge = data.sort_values('last_charge_end_time_ts')


In [5]:
df_store.to_csv('df_store.csv')
df_charge.to_csv('df_charge.csv')

In [6]:

# merge_asof 수행
merged_df = pd.merge_asof(
    df_store,
    df_charge,
    left_on='store_timestamp',
    right_on='last_charge_end_time_ts',
    by=['station_location', 'evse_name'],
    direction='backward',
    tolerance=pd.Timedelta('30min'),
)

merged_df['last_charge_end_time_ts'] = merged_df['last_charge_end_time_ts'].fillna(method='ffill')

print(merged_df)


             store_timestamp evse_type_x station_location  \
0        2024-07-28 11:30:00          FC         CSCS2015   
1        2024-07-28 11:30:00          FC         PW010170   
2        2024-07-28 11:30:00          FC         PW010170   
3        2024-07-28 11:30:00          SC         EV001762   
4        2024-07-28 11:30:00          FC         PW010170   
...                      ...         ...              ...   
19031305 2025-08-17 09:30:00          SC         PIH00593   
19031306 2025-08-17 09:30:00          FC         PIH00593   
19031307 2025-08-17 09:30:00          SC         PIH00593   
19031308 2025-08-17 09:30:00          FC         PI717223   
19031309 2025-08-17 09:30:00          SC         SZAA0032   

         supports_discharge_x  weekday_x  month evse_name  \
0                           n          6      7         0   
1                           n          6      7         6   
2                           n          6      7         7   
3                      

In [7]:
merged_df.to_csv('merged_df.csv')

In [8]:
# 1) unix timestamp 컬럼을 datetime 형식으로 변환
for col in ['connection_start_time_ts', 'connection_end_time_ts', 'charging_start_time_ts', 'charging_end_time_ts']:
    merged_df[col + '_dt'] = pd.to_datetime(merged_df[col], unit='s', errors='coerce')

# 2) 충전 구간이 존재하는 valid_rows (NaN 없는 행)
valid_rows = merged_df.dropna(subset=['connection_start_time_ts_dt', 'connection_end_time_ts_dt']).copy()

# 3) valid_rows를 connection_start_time_ts_dt 기준으로 정렬 (merge_asof 사용을 위해 필수)
valid_rows = valid_rows.sort_values('connection_start_time_ts_dt').reset_index(drop=True)
merged_df_sorted = merged_df.sort_values('store_timestamp').reset_index()

In [9]:
import numpy as np

# 예) 왼쪽 df: merged_df_sorted, 오른쪽 df: valid_rows

# 1단계. merge_asof 전에 left_on 컬럼에 NaN 존재 확인
nan_idx = merged_df_sorted['last_charge_end_time_ts'].isna()

# 2단계. NaN 인 행만 따로 빼서 오른쪽(Valid)에서 groupby key 기준으로 가장 가까운 값을 찾음
import pandas as pd

# 그룹 컬럼
group_cols = ['station_location', 'evse_name']

# NaN인 행만 따로
left_nan_rows = merged_df_sorted[nan_idx]

def fill_na_with_right(row):
    group_mask = (valid_rows['station_location'] == row['station_location']) & (valid_rows['evse_name'] == row['evse_name'])
    candidates = valid_rows.loc[group_mask]
    
    if candidates.empty:
        return np.nan
    
    right_times = candidates['connection_start_time_ts_dt'].dropna()
    if right_times.empty:
        return np.nan
    
    # 위치 기반 인덱싱을 쓰려면 reset_index(drop=True)로 인덱스 재설정
    right_times_reset = right_times.reset_index(drop=True)

    # 현재 NaT인 행에는 store_timestamp나 다른 기준 시간 골랐는지 확인 필요,
    # 예로 store_timestamp 기준으로 가장 가까운 시간 찾기:
    reference_time = row['store_timestamp'] if 'store_timestamp' in row else pd.Timestamp.now()
    
    time_diffs = (right_times_reset - reference_time).abs()
    closest_pos = time_diffs.idxmin()

    closest_time = right_times_reset.iloc[closest_pos]
    
    return closest_time


# 나눠진 NaN 행에 대해 채우기
filled_values = left_nan_rows.apply(fill_na_with_right, axis=1)

# 기존 df에 채우기
merged_df_sorted.loc[nan_idx, 'last_charge_end_time_ts'] = filled_values


In [10]:
merged_df_sorted.to_csv('merged_df.csv')

In [11]:
merged_df_sorted = merged_df_sorted.sort_values('last_charge_end_time_ts').reset_index(drop=True)
valid_rows = valid_rows.sort_values('connection_start_time_ts_dt').reset_index(drop=True)


In [14]:


# 4) merge_asof를 이용하여 store_timestamp가 각 충전 구간 시작 시점 이전 가장 가까운 valid_rows 행과 병합
merged = pd.merge_asof(
    merged_df_sorted,
    valid_rows[['connection_start_time_ts_dt', 'connection_end_time_ts_dt'] + 
                ['connection_start_time_ts', 'connection_end_time_ts',
                 'charging_start_time_ts', 'charging_end_time_ts',
                 'expected_departure_time_ts','expected_departure_time_missing',
                 'idle_time_ts','expected_usage_duration_ts','expected_time_diff_missing','actual_usage_duration_ts',
                 'actual_charging_duration_ts','actual_charging_duration_missing','start_delay_duration_ts',
                 'start_delay_duration_missing','post_charge_departure_delay_ts','post_charge_departure_delay_missing','usage_departure_time_diff_ts',
                 'usage_departure_time_diff_missing','delivered_kwh','requested_kwh','kwh_request_diff','kwh_per_usage_time',
                 'charging_start_time_missing', 'charging_end_time_missing', 'duration_per_kwh_missing', 'kwh_per_usage_time_missing',
                 'station_location', 'evse_name','scheduled_charge', 'usage_departure_range','post_charge_departure_range','cluster']],
    left_on='last_charge_end_time_ts',
    right_on='connection_start_time_ts_dt',
    by=['station_location', 'evse_name'],
    direction='backward',
    suffixes=('', '_filled')
)

# 5) 병합된 결과에 대해 store_timestamp가 충전 구간의 끝시간(connection_end_time_ts_dt) 이후인 경우 매칭 실패로 간주해서 _filled 컬럼 모두 NaN으로 처리
mask_out_of_range = merged['store_timestamp'] > merged['connection_end_time_ts_dt_filled']
for col in merged.columns:
    if col.endswith('_filled'):
        merged.loc[mask_out_of_range, col] = np.nan

# 6) NaN 값에 대해서만 '_filled' 접미사 컬럼에서 값을 채우는 함수 적용
fill_cols = [
    'connection_start_time_ts', 'connection_end_time_ts',
    'charging_start_time_ts', 'charging_end_time_ts',
    'expected_departure_time_ts','expected_departure_time_missing',
    'idle_time_ts','expected_usage_duration_ts','expected_time_diff_missing','actual_usage_duration_ts',
    'actual_charging_duration_ts','actual_charging_duration_missing','start_delay_duration_ts',
    'start_delay_duration_missing','post_charge_departure_delay_ts','post_charge_departure_delay_missing','usage_departure_time_diff_ts',
    'usage_departure_time_diff_missing','delivered_kwh','requested_kwh','kwh_request_diff','kwh_per_usage_time',
    'charging_start_time_missing', 'charging_end_time_missing', 'duration_per_kwh_missing', 'kwh_per_usage_time_missing',
    'scheduled_charge', 'usage_departure_range','post_charge_departure_range','cluster',
    'connection_start_time_ts_dt','connection_end_time_ts_dt','charging_start_time_ts_dt','charging_end_time_ts_dt'
]

for col in fill_cols:
    filled_col = col + '_filled' if col+'_filled' in merged.columns else None
    if filled_col:
        merged[col] = merged[col].combine_first(merged[filled_col])

# 7) 필요하다면 index 원복
merged = merged.sort_index()

# 8) 결과 확인
data_filled = merged.drop(columns=[c for c in merged.columns if c.endswith('_filled')])

# 9) 필요시 datetime 컬럼을 unix timestamp로 변환하는 함수 사용
def datetime_to_unix(dt):
    if pd.isna(dt):
        return np.nan
    else:
        return int(dt.value // 10**9)

# 예) data_filled['some_datetime_col_unix'] = data_filled['some_datetime_col'].apply(datetime_to_unix)

In [15]:
data_filled.to_csv('data_full.csv',index=False)